# Projet ML for Econometrics
## L'échec scolaire dans le supérieur et l'effet causal des bourses d'études

## Introduction

Ce notebook est le rapport pour le projet réalisé dans le cadre du projet pour le cours de ML for Econometrics présenté par Mr. Doutreligne et Mr. Crépon.

Ce rapport a été rédigé par
Etienne Chastel;
Camille Frouard;
Enzo Guebli;
Gabriel Orsatti;
Raphaël Zambelli--Palacio

## Plan

#### Nous présentons dans un premier temps la base de données, notre problématique et justifions notre recherche. Nous présentons ensuite la base de données lors d'une analyse descriptive. Nous appliquons enfin diverses méthodes pour identifier l'effet causal recherché et testons la robustesse de notre étude, avant de discusuter des implications de nos résultats.

## Présentation de la base de données et de la problématique

Nous nous appuyons sur le dataset "Predict Students' Dropout and Academic Success" qui est une base de données disponible publiquement via ce [lien](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success).

Cette base de données répertorie le parcours scolaire de 4424 étudiants dans différents cursus (agronomie, design, infirmier, etc.), inscrits à la Polytechnic University of Portalegre au Portugal. La BDD donne pour chaque élève de nombreuses informations socio-démographiques, des données sur son parcours scolaire avant et après son entrée dans l'enseignement supérieur, des informations sur la situation economique du pays l'année de l'entrée à l'université, ainsi que la situation de l'élève à la fin de la période scolaire.

Cette dernière variable est ici notre variable d'intérêt, elle peut prendre trois valeurs différentes : _dropout_ si l'élève a abandonné les études, _graduate_ si l'élève a été diplômé, ou _enrolled_ si l'élève continue les cours dans un temps supplémentaire.

Notre problématique est la suivante : Pouvons nous identifier un effet causal de l'obtention d'une bourse d'étude sur l'obtention d'un diplôme ?

## Littérature

Nous nous intéressons ici au rôle que joue l'obtention d'une bourse d'étude lors de l'entrée dans l'enseignement supérieur sur le succès académique de l'élève.

Les bourses pour l'univeristé Polytechnique de Portalegre sont délivrées sur critères sociaux tels que définis par le gouvernement portugais. Un nombre faible de bourses au mérite est délivré (6 seulement en 2023-2024) (Politécnico de Portalegre, 2026). Les bourses délivrées sur critères sociaux sont associées dans la littérature à un taux de succès plus élevé des étudiant.e.s, c'est le cas dans des études réalisées au Pakistan (Ahmed et al. 2024), aux Etats-Unis (Herzog 2005) et au Portugal (Casanova et al. 2023). Néanmoins l'étude de Herzog (2005) montre la persistance des inégalités. Les aides pour les élèves de milieux sociaux défavorisés ne  permettent pas à long terme de combler leur écart de chance de rester dans l'université comparés aux étudiants favorisés, disposant d'un capital économique, social et culturel plus élevé. 

Dans ce contexte, notre étude cherche à identifier l'effet causal d'une bourse d'étude sur le succès des élèves, afin d'estimer l'effet de la bourse seule. Nous prenons ici l'entrée à l'université comme une expérience, dont le traitement est de recevoir une bourse d'étude ou non, et nous étudions son effet sur la réussite scolaire dans le supérieur.

#### Références : 

Ahmed, R., Ahmed, A., Barkat, W., & Ullah, R. (2024). Impact of scholarships on student success: a case study of the University of Turbat, Pakistan. _The Pakistan Development Review_, 231–258. https://doi.org/10.30541/v61i2pp.231-258 

Casanova, J. R., Castro-López, A., Bernardo, A. B., & Almeida, L. S. (2023). The Dropout of First-Year STEM Students: Is It Worth Looking beyond Academic Achievement? _Sustainability_, 15(2), 1253. https://doi.org/10.3390/su15021253 

Herzog, S. (2005). Measuring determinants of student Return VS. Dropout/Stopout VS. transfer: A First-to-Second Year analysis of new freshmen. _Research in Higher Education_, 46(8), 883–928. https://doi.org/10.1007/s11162-005-6933-7 

Politécnico de Portalegre. (2026). _Bolsas de Estudo - Politécnico de Portalegre_. https://www.ipportalegre.pt/pt/estudantes/servicos-de-acao-social/bolsas-de-estudo/

## Formulation PICO de notre problématique

P : La population étudiée ici est un ensemble d'étudiants en études entre 2008 et 2018

I : L'intervention que l'on considère ici est le fait de recevoir une bourse durant ses études

C : On compare cela avec les étudiants n'ayant pas reçu de bourse durant leur études

O : L'outcome considéré est le fait d'obtenir son diplôme à la fin du parcours scolaire

## Variables et DAG

Nos variables d'étude sont les suivantes :
- Variables démographiques : Âge à l'entrée à l'université, genre, statut marital, nationalité, statut international, statut déplacé
- Variables socio-économiques : Niveau d'éducation du père et de la mère, profession du père et de la mère, niveau d'éducation de l'étudiant avant son entrée à l'université, statut de besoin spéciaux d'éducation
- Variables de déroulement économique des études : statut d'endettement, statut de paiement des frais de scolarité
- Variables de statut académiques : Place de classement de l'université parmis les voeux, mode de candidature, filière, note d'admission, statut d'étude en journée ou cours du soir, nombre d'examens nationaux réalisés, note moyenne aux examens nationaux
- Variables de déroulement académique des études : nombre de crédits où l'élève est inscrit, évalué et crédité au premier et deuxième semestre. Nombre d'évaluations et note moyenne au premier et deuxième semestre.
- Variables de contexte économique : Taux de chômage, taux d'inflation et PIB du pays

- Variable d'intérêt : Statut scolaire (abandon scolaire sans avoir obtenu de diplôme, obtention en cours, diplôme obtenu)
- Variable de traitement : détenteur d'une bourse

Nous pouvons ainsi tracer le DAG, afin de spécifier nos hypothèses en terme d'effets causaux et de nous assurer de l'absence de colliding variables

In [ ]:
import os, urllib.request
from matplotlib import font_manager
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as path_effects
import seaborn as sns

# --- Configuration  ---
FONT_TITLE = 'Playfair Display'
FONT_BODY  = 'Source Sans 3'
BG_COLOR   = '#FAF7F2'
TEXT_COLOR = '#2C2824'
GRID_COLOR = '#E8E4DD'

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("\n Construction du DAG (clean layout)")

fig, ax = plt.subplots(figsize=(18, 11))
ax.set_xlim(0, 15)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# ======================
# NODES
# ======================
nodes = {
    'SES': (1.5, 9.0, 'SES parents', '#4caf84', 'black'),
    'ACAD0': (1.5, 6.5, 'Académique initial', '#4caf84', 'black'),
    'DEM': (1.5, 4.0, 'Démographie', '#4caf84', 'black'),
    'INST': (5.5, 9.5, 'Choix institutionnel', '#90a4ae', 'black'),
    'U': (7.5, 9.5, 'Motivation initiale\n(non observé)', '#ff7043', 'white'),
    'U2': (13.5, 9.5, 'Motivation en cours\n(non observé)', '#ff7043', 'white'),
    'D': (7.5, 4.8, 'Bourse', '#1976D2', 'white'),
    'MACRO': (3.5, 1.0, 'Macro', '#f5a623', 'black'),
    'DEBT': (11.0, 10, 'Endettement', '#9c27b0', 'white'),
    'M': (11.0, 8.5, 'Performance', '#9c27b0', 'white'),
    'C': (11.0, 7.0, 'Crédits', '#9c27b0', 'white'),
    'FEES': (11.0, 5.5, 'Frais', '#9c27b0', 'white'),
    'Y': (13.5, 2, 'Statut scolaire', '#e05c5c', 'white'),
}
for key, (x, y, label, color, tcolor) in nodes.items():
    circle = plt.Circle((x, y), 0.85, color=color, zorder=3, alpha=0.95)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=9, color=tcolor, fontweight='bold')

# ======================
# Edges
# ======================
def draw_arrow(src, dst):
    sx, sy = nodes[src][0], nodes[src][1]
    dx, dy = nodes[dst][0], nodes[dst][1]

    if src in ['DEBT', 'FEES', 'M', 'C'] or (src == 'D' and dst in ['DEBT','FEES','M','C']):
        color, style = '#9c27b0', 'dashed'
    elif src in ['U', 'U2']:
        color, style = '#ff7043', 'dotted'
    elif src == 'MACRO':
        color, style = '#f5a623', 'solid'
    else:
        color, style = '#424242', 'solid'

    rad = 0
    if dst == 'Y' and src in ['SES', 'ACAD0', 'DEM']:
        rad = 0.25
    if src in ['U', 'U2']:
        rad = 0.35
    if src in ['DEBT', 'M', 'C', 'FEES'] and dst == 'Y':
        rad = -0.15
    if src == 'INST' and dst == 'Y':
        rad = 0.2

    ax.annotate(
        "",
        xy=(dx, dy), xytext=(sx, sy),
        arrowprops=dict(
            arrowstyle='->',
            color=color,
            lw=2,
            linestyle=style,
            connectionstyle=f"arc3,rad={rad}",
            shrinkA=28,
            shrinkB=28
        )
    )

arrows = [
    ('SES', 'D'), ('ACAD0', 'D'), ('DEM', 'D'), ('MACRO', 'D'),
    ('SES', 'Y'), ('ACAD0', 'Y'), ('DEM', 'Y'), ('MACRO', 'Y'),

    ('SES', 'ACAD0'),
    ('DEM', 'ACAD0'),
    ('DEM', 'INST'),
    ('SES', 'INST'),
    ('ACAD0', 'INST'),

    ('INST', 'D'),
    ('INST', 'Y'),

    ('D', 'Y'),

    ('D', 'DEBT'),
    ('D', 'FEES'),
    ('D', 'M'),
    ('D', 'C'),

    ('DEBT', 'Y'),
    ('FEES', 'Y'),
    ('M', 'Y'),
    ('C', 'Y'),

    ('C', 'M'),

    ('U', 'D'),
    ('U', 'Y'),
    ('U', 'M'),
    ('U', 'C'),
    ('U', 'U2'),

    ('M', 'U2'),
    ('U2', 'Y')
]

for src, dst in arrows:
    draw_arrow(src, dst)

# ======================
# LEGENDE
# ======================
legend_items = [
    mpatches.Patch(color='#4caf84', label='Confounders'),
    mpatches.Patch(color='#1976D2', label='Traitement'),
    mpatches.Patch(color='#e05c5c', label='Outcome'),
    mpatches.Patch(color='#9c27b0', label='Médiateurs'),
    mpatches.Patch(color='#90a4ae', label='Choix initiaux'),
    mpatches.Patch(color='#f5a623', label='Macro'),
    mpatches.Patch(color='#ff7043', label='Non observé'),
]

ax.legend(handles=legend_items, loc='lower right', fontsize=10, framealpha=0.95)

ax.set_title(
    "DAG - Effet causal de la bourse sur le statut scolaire",
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('dag_causal_clean.png', bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

## Analyse descriptive des données

L'objectif de cette seconde partie est de visualiser en détails la base de données. Nous nous attarderons particulièrement sur l'outcome et sur les variables corrélées avec le fait d'obtenir une bourse.

### Téléchargement des données et nettoyage de la BDD

In [ ]:
!pip install ucimlrepo

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo 


### Configuration du style visuel

La cellule suivante télécharge les polices Playfair Display et Source Sans 3 puis applique une palette cohérente à tous les graphiques du notebook.

In [ ]:
# Configuration de l'esthétique et couleurs des graphiques. 
PALETTE_TARGET = {'Dropout': '#D4626E', 'Enrolled': '#F0B87E', 'Graduate': '#8CBF8C'}
PALETTE_BOURSE = {
    'Boursier': '#7EADD4', 'Non-Boursier': '#C4BFB6',
    0: '#C4BFB6', 1: '#7EADD4',
}
COLORS = {
    'treated': '#7EADD4', 'control': '#C4BFB6',
    'effect': '#5B8DB8', 'dropout': '#D4626E',
    'graduate': '#8CBF8C', 'neutral': '#A89F96',
    'placebo': '#C4BFB6', 'accent': '#D4626E',
}
TARGET_ORDER = ['Dropout', 'Enrolled', 'Graduate']

plt.rcParams.update({
    'figure.facecolor': BG_COLOR, 'axes.facecolor': BG_COLOR,
    'axes.edgecolor': GRID_COLOR, 'axes.labelcolor': TEXT_COLOR,
    'text.color': TEXT_COLOR, 'xtick.color': TEXT_COLOR, 'ytick.color': TEXT_COLOR,
    'grid.color': GRID_COLOR, 'grid.alpha': 0.5,
    'font.family': FONT_BODY, 'font.size': 11,
    'axes.titlesize': 15, 'axes.labelsize': 12,
    'figure.dpi': 150,
    'axes.spines.top': False, 'axes.spines.right': False,
})
sns.set_theme(style='white', rc={
    'axes.facecolor': BG_COLOR,
    'figure.facecolor': BG_COLOR,
    'grid.color': GRID_COLOR,
})
print("Style visuel configuré")


In [ ]:
print("--- 1. Chargement du dataset ---")
dataset = fetch_ucirepo(id=697) 
X = dataset.data.features.copy()
y = dataset.data.targets 
X["Application mode"] = X["Application mode"].astype(str)
X["Course"] = X["Course"].astype(str)
X["Father\'s occupation"] = X["Father\'s occupation"].astype(str)
X["Mother\'s occupation"] = X["Mother\'s occupation"].astype(str)
X["Father\'s qualification"] = X["Father\'s qualification"].astype(str)
X["Mother\'s qualification"] = X["Mother\'s qualification"].astype(str)
X["Marital Status"] = X["Marital Status"].astype(str)
X["Nacionality"] = X["Nacionality"].astype(str)
X["Previous qualification"] = X["Previous qualification"].astype(str)
df = pd.concat([X, y], axis=1)

print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes\n")

print("--- 2. Types de variables et Valeurs manquantes ---")
valeurs_manquantes = pd.DataFrame({
    'Type': df.dtypes,
    'Manquants (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Valeurs Uniques': df.nunique()
})
print(valeurs_manquantes)

print("\n--- 3. Statistiques Descriptives ---")
display(df.describe().T.round(2))

Comme nous le montrent ces résultats, la base de données ne contient pas de valeur manquante.

Cependant, ce résultat cache d'une certaine façon la réalité : Rappelons que dans note BDD, certains élèves abandonnent leurs études au cours de l'année.
Ainsi, les variables inquant le nombre d'ECTS crédités par semestre ainsi que les notes des élèves n'ont pas réellement de signification : en effet, un élève qui abandonne son année aura administrativement des notes nulles pour le reste de l'année.

Comme l'idée de notre projet est d'étudier les facteurs qui influent en amont l'obtention d'une bourse et la réussite de son année, ces variables en question ne semblent pas pertinentes et risquent de nous induire en erreur. Ainsi, on les supprime du dataset.

Notons également que certaines variables, comme la profession des parents ou la nationalité de l'élève, sont des variables nominales encodées en entier. Il est donc important de les transformer en string (chaîne de caractères) pour ne pas faire d'erreur par la suite.

In [ ]:
df["Mother\'s occupation"].value_counts()

Lorsque l'on regarde les variables indiquant les métiers des parents de l'élève, on note que cela correspond plus ou moins au code ISCO (nomenclature européenne des métiers). Aux détails près que certains individus sont associés avec un métier plus précis (2 digits), pour les reconnaître, ce sont ceux qui commençent par un "1".

Comme cela ne représente que très peu d'individus, on agrège l'information au premier digit (ainsi 123 qui correspond au métier de professeur (teacher, code ISCO 23) est compris dans la section 2 : professionnels).

In [ ]:
# Suppression des colonnes inutiles
data_propre = df.drop([

    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
    
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)"

], axis=1)

# Pour les variables d'occupation des parents, on agrège au premier chiffre de la nomenclature ISCO
def agrege_occupation(row):
    s = row
    if isinstance(s, str) and len(s) == 3:
        return s[1]
    else:
        return s

data_propre["Father_occupation_sector"] = data_propre["Father\'s occupation"].apply(agrege_occupation)
data_propre["Mother_occupation_sector"] = data_propre["Mother\'s occupation"].apply(agrege_occupation)


data_propre = data_propre.drop(["Father\'s occupation","Mother\'s occupation"], axis=1)
data_propre["Father_occupation_sector"] = data_propre["Father_occupation_sector"].map({
    "0": "Student",
    "1": "Managers",
    "2": "Professionals",
    "3": "Technicians",
    "4": "Clerical Support Workers",
    "5": "Services and Sales Workers",
    "6": "Skilled Workers",
    "7": "Craft Workers",
    "8": "Plant Operators",
    "9": "Elementary Occupations",
    "10": "Armed Forces profession",
    "90": "Other situation",
    "99": "No response",
    })
    
data_propre["Mother_occupation_sector"] = data_propre["Mother_occupation_sector"].map({
    "0": "Student",
    "1": "Managers",
    "2": "Professionals",
    "3": "Technicians",
    "4": "Clerical Support Workers",
    "5": "Services and Sales Workers",
    "6": "Skilled Workers",
    "7": "Craft Workers",
    "8": "Plant Operators",
    "9": "Elementary Occupations",
    "10": "Armed Forces profession",
    "90": "Other situation",
    "99": "No response",
    })


# On renomme les valeurs de certaines variables catégorielles pour une meilleure compréhension des données
data_propre["Course"] = data_propre["Course"].map({
    "33": "Biofuel Production",
    "171": "Animation Design",
    "8014": "Social Service (evening)",
    "9003": "Agronomy",
    "9070": "Communiation Design",
    "9085": "Veterinary",
    "9119": "Informatics",
    "9130": "Equiculture",
    "9147": "Management",
    "9238": "Social Service",
    "9254": "Tourism",
    "9500": "Nursing",
    "9556": "Oral Hygiene",
    "9670": "Marketing",
    "9773": "Journalism",
    "9853": "Education",
    "9991": "Management (evening)",
    })

data_propre["Daytime/evening attendance"] = data_propre["Daytime/evening attendance"].map({
    1: "Daytime",    
    0: "Evening",
    })

data_propre["Gender"] = data_propre["Gender"].map({
    1: "Male",    
    0: "Female",
    })

data_propre["Scholarship holder"] = data_propre["Scholarship holder"].map({
    1: "Boursier",    
    0: "Non-Boursier",
    })

data_propre["International"] = data_propre["International"].map({
    1: "Yes",    
    0: "No",
    })


categorie_graduate = ['Dropout', 'Enrolled', 'Graduate']
data_propre['Target'] = pd.Categorical(data_propre['Target'], categories=categorie_graduate, ordered=True)


In [ ]:
data_propre.describe(include="all").T

### Statistiques Descriptives :

#### La répartition du dropout selon les caractéristiques des individus

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG_COLOR)
counts = data_propre['Target'].value_counts().reindex(TARGET_ORDER)
palette = [PALETTE_TARGET[t] for t in TARGET_ORDER]
bars = ax.bar(counts.index, counts.values, color=palette, width=0.55,
              edgecolor=BG_COLOR, linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 25,
            f'{val:,}\n({val/counts.sum()*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, color=TEXT_COLOR)
ax.set_title('Répartition des étudiants par statut académique',
             fontsize=16, fontfamily=FONT_TITLE, pad=14)
ax.set_xlabel('Statut', fontsize=12, labelpad=8)
ax.set_ylabel('Nombre d\'étudiants', fontsize=12, labelpad=8)
ax.set_facecolor(BG_COLOR)
ax.grid(axis='y', color=GRID_COLOR, alpha=0.6, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


On note ainsi que les trois catégories de notre outcome ne sont pas pas présentes dans les mêmes proportions dans ce dataset. Si une majorité des élèves sont diplômés à la fin de l'année, un tiers des élèves abandonnent les études durant l'année. La dernière catégorie est constituée des élèves qui n'ont pas validé dans le temps réglementaire, et qui sont donc encore inscrits pour finir leur cursus. Cette situation concerne un peu moins de 18% des élèves.

In [ ]:
cross_tab = pd.crosstab(data_propre['Course'], data_propre['Target'])
cross_tab = cross_tab.sort_values(by='Graduate')[TARGET_ORDER]
cross_tab_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

bottom = pd.Series([0]*len(cross_tab_pct), index=cross_tab_pct.index)
for status in TARGET_ORDER:
    ax.barh(cross_tab_pct.index, cross_tab_pct[status], left=bottom,
            color=PALETTE_TARGET[status], label=status, edgecolor=BG_COLOR, linewidth=0.5)
    bottom += cross_tab_pct[status]

ax.set_title('Répartition du statut académique par filière (normalisée)',
             fontsize=15, fontfamily=FONT_TITLE, pad=12)
ax.set_xlabel('Proportion', fontsize=12, labelpad=8)
ax.legend(loc='lower right', fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
ax.set_xlim(0, 1)
ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


On remarque ici que certaines études semblent plus "simples" que d'autres. Par exemple, les études d'infirmier possèdent un taux de réussite bien plus élevé que les études d'informatique (72% contre 8% de diplômés). Ces écarts ne peuvent pas être immédiatement interprétés comme reflétant des niveaux de difficulté différents : ils pourraient également provenir de différence dans la composition des étudiants - les élèves en études d'infirmier pouvant par exemple être nettement plus motivés que ceux en études d'informatique dès l'entrée en formation.  

De plus, il est important de noter que les effectifs sont très hétérogènes entre les différentes études dans notre échantillon. Les études d'infirmier comptabilisent presque 800 étudiants, tandis que les études maxillo-faciales ne comptent que 86 étudiants au sein de notre BDD. 

In [ ]:
cross_tab = pd.crosstab(data_propre['Father_occupation_sector'], data_propre['Target'])
cross_tab = cross_tab.div(cross_tab.sum(axis=1), axis=0)
cross_tab = cross_tab.sort_values(by='Graduate')[TARGET_ORDER]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

bottom = pd.Series([0]*len(cross_tab), index=cross_tab.index)
for status in TARGET_ORDER:
    ax.barh(cross_tab.index, cross_tab[status], left=bottom,
            color=PALETTE_TARGET[status], label=status, edgecolor=BG_COLOR, linewidth=0.5)
    bottom += cross_tab[status]

ax.set_title('Réussite académique selon la CSP du père (normalisée)',
             fontsize=15, fontfamily=FONT_TITLE, pad=12)
ax.set_xlabel('Proportion', fontsize=12, labelpad=8)
ax.legend(loc='lower right', fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
ax.set_xlim(0, 1)
ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


On observe ci-dessus la proportion de réussite et d'abandon selon la catégorie socio-professionnelle du père. Il apparaît que les élèves ayant un père toujours en étude ont en majorité abandonné les études (61%). Ce pourcentage est le double par rapport aux autres CSP du père, où la proportion d'abandon est d'environ 31%.

Ce qui est surprenant, c'est de constater que, quelle que soit la profession du père, les taux d'abandon sont sensiblement identiques, alors même que l'on pourrait s'attendre à avoir des taux plus faibles pour les étudiants ayant des parents qualifiés. La réalité de nos données penche plutôt dans le sens contraire : si le taux d'abandon est de 36% pour les fils et filles de cadres, cette proportion descend autour de 28% pour les employés moins qualifiés et les ouvriers.

A nouveau, il importe de ne pas sur-interpréter ces statistiques. Les fils et filles de cadres peuvent entreprendre des études plus difficiles, ou plus longues, que les fils et filles d'ouvriers. Par ailleurs, l'abandon peut prendre une signification différente pour ces deux populations: entrée sur le marché du travail avec une faible qualification pour les uns, réorientation vers d'autres études pour les autres. On sait en effet par plusieurs travaux de sociologie (par exemple, _Les Héritiers_, Bourdieu et Passeron, 1964) que les étudiants issus des classes ouvrières et classes moyennes envisagent les études comme un manière de progresser socialement et économiquement, tandis que les étudiants issus des classes favorisées se déterminent plus par leur plaisir, leur attirance pour telles ou telles disciplines. 

#### Quelles différences entre les étudiants boursiers et les non-boursiers ?

In [ ]:
data_boursiers     = data_propre[data_propre['Scholarship holder'] == 'Boursier']
data_non_boursiers = data_propre[data_propre['Scholarship holder'] == 'Non-Boursier']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Statut académique : Boursiers vs Non-boursiers',
             fontsize=16, fontfamily=FONT_TITLE, y=1.02)

for ax, data, label, color in [
    (axes[0], data_boursiers, 'Boursiers', COLORS['treated']),
    (axes[1], data_non_boursiers, 'Non-Boursiers', COLORS['control'])
]:
    ax.set_facecolor(BG_COLOR)
    counts = data['Target'].value_counts().reindex(TARGET_ORDER)
    colors = [PALETTE_TARGET[t] for t in TARGET_ORDER]
    bars = ax.bar(counts.index, counts.values, color=colors,
                  width=0.55, edgecolor=BG_COLOR, linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{val/counts.sum()*100:.1f}%',
                ha='center', va='bottom', fontsize=10, color=TEXT_COLOR)
    ax.set_title(label, fontsize=14, fontfamily=FONT_TITLE)
    ax.set_ylabel('Nombre d\'étudiants', fontsize=11)
    ax.grid(axis='y', color=GRID_COLOR, alpha=0.5)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


Le taux d'abandon est nettement plus élevé pour les étudiants non-boursiers que pour les boursiers. Mais ici, l'interprétation causale de l'effet une bourse sur le décrochage scolaire est exclue. En effet, comme nous allons le voir, les étudiants boursiers et non-boursiers n'ont pas du tout les mêmes caractéristiques. Nous ne sommes en aucun cas dans un contexte de RCT où la bourse aurait été attribuée aléatoirement au sein de la population.

In [ ]:
cross_tab = pd.crosstab(data_propre['Father_occupation_sector'],
                         data_propre['Scholarship holder'])
cross_tab = cross_tab.div(cross_tab.sum(axis=1), axis=0)
cross_tab = cross_tab.sort_values(by='Non-Boursier')

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

bottom = pd.Series([0]*len(cross_tab), index=cross_tab.index)
for cat, color in [('Non-Boursier', COLORS['control']), ('Boursier', COLORS['treated'])]:
    if cat in cross_tab.columns:
        ax.barh(cross_tab.index, cross_tab[cat], left=bottom,
                color=color, label=cat, edgecolor=BG_COLOR, linewidth=0.5)
        bottom += cross_tab[cat]

ax.set_title('Proportion de boursiers selon la CSP du père',
             fontsize=15, fontfamily=FONT_TITLE, pad=12)
ax.set_xlabel('Proportion', fontsize=12, labelpad=8)
ax.legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
ax.set_xlim(0, 1)
ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


En toute logique, les boursiers ont des parents moins aisés que les autres, et sont donc surreprésentés dans les CSP les moins qualifiées. 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Figure 3 – Profil socio-économique : Boursiers vs Non-boursiers', 
             fontsize=18, fontweight='bold', fontfamily=FONT_TITLE, color=TEXT_COLOR, y=1.02)

kde_plots = [
    (axes[0, 0], 'Age at enrollment', "Âge à l'inscription"),
    (axes[0, 1], 'Admission grade', "Note d'admission"),
    (axes[1, 0], 'Previous qualification (grade)', "Précédente note (avant l'étude)")
]

for ax, var, title in kde_plots:
    ax.set_facecolor(BG_COLOR)
    
    for val, label, color in [('Non-Boursier', 'Non-Boursier', COLORS['control']), 
                               ('Boursier', 'Boursier', COLORS['treated'])]:
        subset = data_propre[data_propre['Scholarship holder'] == val][var].dropna()
        sns.kdeplot(subset, ax=ax, fill=True, alpha=0.45, color=color, label=label, linewidth=2)
    
    # Esthétique des axes
    ax.set_title(title, fontsize=14, fontfamily=FONT_TITLE, fontweight='bold')
    ax.set_xlabel(var, fontsize=11, fontfamily=FONT_BODY)
    ax.set_ylabel('Densité', fontsize=11, fontfamily=FONT_BODY)
    ax.grid(color=GRID_COLOR, alpha=0.4, linestyle='-')
    ax.set_axisbelow(True)
    ax.legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)

ax_bar = axes[1, 1]
ax_bar.set_facecolor(BG_COLOR)

gender_cross = pd.crosstab([data_propre['Gender'], data_propre['Scholarship holder']], 
                            data_propre['Target'], normalize='index') * 100
gender_cross = gender_cross.reindex(columns=TARGET_ORDER)
labels = ['Femme / Boursier', 'Femme / Non-Boursier', 'Homme / Boursier', 'Homme / Non-Boursier']
gender_cross.index = labels

gender_cross.plot(kind='barh', stacked=True, ax=ax_bar,
                  color=[PALETTE_TARGET[t] for t in TARGET_ORDER], alpha=0.9)

ax_bar.set_title("Statut par Genre × Statut boursier (%)", fontsize=14, fontfamily=FONT_TITLE, fontweight='bold')
ax_bar.set_xlabel("Pourcentage", fontsize=11, fontfamily=FONT_BODY)
ax_bar.grid(axis='x', color=GRID_COLOR, alpha=0.4)
ax_bar.set_axisbelow(True)

ax_bar.legend(title='Statut', loc='lower right', fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
for container in ax_bar.containers:
    ax_bar.bar_label(container, fmt='%.0f%%', label_type='center', 
                     fontsize=9, fontfamily=FONT_BODY, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

Les estimations des densités présentées ci-dessus révèlent des différences importantes dans la composition des deux groupes, non-boursiers et boursiers. Les premiers entrent en étude à un plus jeune âge que les seconds. D'autre part, la variance de niveau à l'entrée, capturée par les notes d'admission, est plus importante chez les boursiers qui sont plus fréquents à des niveaux plus faibles et des niveaux plus élevés de notes que les non-boursiers. Cela nous conduit à une hypothèse d'hétérogénéité des boursiers, composés d'étudiants très motivés obtenant de bons résultats scolaires, et d'étudiants avec de plus grandes difficulté scolaire. 

Enfin, la variable genre joue dans le même sens au sein de chaque groupe : les femmes abandonnent nettement moins leurs études que les hommes, et sont proportionnellement nettement plus nombreuses à diplômer. 

## Modèle Double Post-Lasso

Comme nous l'avons vu, une simple régression de $Y$ (succès scolaire) sur $D$ (bourse) et $X$ (autres covariables) ne permet pas d'estimer l'effet causal de la bourse sur le décrochage : des variables omises (motivation, capital social) confondent à la fois la probabilité d'obtenir une bourse et le risque de dropout.

#### Pourquoi le Double Post-Lasso ?

La méthode du **Double Post-Lasso** (Belloni, Chernozhukov & Hansen, 2014, *Review of Economic Studies*) répond au problème de sélection de variables dans les modèles à haute dimension. Dans notre dataset, le nombre de covariables après encodage des variables catégorielles dépasse facilement $p > 100$ - un régime où la sélection manuelle est impossible et où une régression OLS ordinaire souffre d'overfitting.

L'idée centrale repose sur le **Partially Linear Model (PLM)** :
$$Y = \theta D + g(X) + \varepsilon, \quad \mathbb{E}[\varepsilon | D, X] = 0$$

où $g(X)$ capture les effets non-paramétriques des covariables. Dans la version Post-Lasso, $g(X)$ est approchée par un modèle linéaire sparse sélectionné par Lasso.

#### Procédure en trois étapes

La procédure implémentée ici suit exactement les trois étapes décrites par Belloni et al. (2014) :

1. **Régression Lasso de $Y$ sur $X$** (sans $D$) : sélection des covariables qui prédisent l'outcome. On retient les variables avec coefficient $\neq 0$, puis on fait une régression logistique Post-Lasso pour obtenir les résidus $\tilde{Y} = Y - \hat{\ell}(X)$.

2. **Régression Lasso de $D$ sur $X$** : sélection des covariables qui prédisent le traitement (propensity). Régression logistique Post-Lasso → résidus $\tilde{D} = D - \hat{m}(X)$.

3. **Régression des résidus** : OLS de $\tilde{Y}$ sur $\tilde{D}$. Le coefficient obtenu est l'estimateur $\hat{\theta}$ de l'effet causal.

#### Propriété de Neyman-orthogonalité

La raison pour laquelle cette procédure fonctionne est que le score correspondant est **Neyman-orthogonal** par rapport aux nuisances $\ell_0(X)$ et $m_0(X)$ : une erreur de premier ordre dans leur estimation n'affecte pas $\hat{\theta}$ au premier ordre (Chernozhukov et al., 2018). Cela garantit la $\sqrt{n}$-normalité de l'estimateur même quand les nuisances sont estimées à une vitesse plus lente.

> Le Double Post-Lasso est une version restreinte du DML où (i) les nuisances sont estimées par Lasso sur l'ensemble de l'échantillon sans cross-fitting et (ii) l'hypothèse de sparsité doit être satisfaite. Dans la section suivante, on relâchera ces deux contraintes.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

data_propre_copy = data_propre.copy()

variables_categorielles = ["Displaced","Educational special needs","Debtor","Tuition fees up to date","Daytime/evening attendance","Gender","International",'Application mode', 'Course', 'Father\'s qualification', 'Mother\'s qualification',"Marital Status","Nacionality","Previous qualification","Father_occupation_sector","Mother_occupation_sector"]
data_dummies = pd.get_dummies(data_propre_copy, columns=variables_categorielles, drop_first=True)

# Variables de contrôle (X), traitement (D), outcome (Y)
X = data_dummies.drop(columns=['Target', 'Scholarship holder'])

X = sm.add_constant(X)

D = (data_dummies['Scholarship holder'] == 'Boursier').astype(int)
Y = (data_dummies['Target'] == 'Dropout').astype(int)

variables_numeriques = ['Application order', 'Previous qualification (grade)', 'Admission grade',"Unemployment rate","Inflation rate","GDP"]
scaler = StandardScaler()
X[variables_numeriques] = scaler.fit_transform(X[variables_numeriques])
X = X.map(lambda x: int(x) if isinstance(x, bool) else x)

### Régression Post-Lasso sur $Y$ (outcome)

On régresse l'outcome $Y$ (1 si _dropout_, 0 sinon) sur les covariables $X$ via un **LassoCV** (5 folds). Le Lasso impose une pénalité $\ell_1$ sur les coefficients :
$$\hat{\beta}^{Lasso} = \arg\min_{\beta} \left\{ \frac{1}{n}\|Y - X\beta\|^2 + \lambda \|\beta\|_1 \right\}$$

Ce faisant, certains coefficients sont exactement mis à zéro, réalisant une **sélection automatique** des variables pertinentes. Le paramètre $\lambda$ est choisi par validation croisée pour minimiser l'erreur de prédiction.

Pourquoi utiliser une **régression logistique Post-Lasso** plutôt que de garder directement les prédictions Lasso ? Parce que le Lasso introduit un biais de régularisation dans les coefficients. En réestimant par OLS (ou logit) sur les variables sélectionnées uniquement, on récupère des estimateurs sans biais - c'est la logique du *Post-Lasso* (Belloni & Chernozhukov, 2013).


In [ ]:
# Régression Lasso avec validation croisée de Y sur X
lasso_Y = LassoCV(cv=5, random_state=42).fit(X, Y)

lasso_Y_coef = pd.Series(lasso_Y.coef_, index=X.columns)
print("Coefficients Lasso :\n", lasso_Y_coef)

variables_selectionnees = lasso_Y_coef[lasso_Y_coef != 0].index.tolist()
print("\nFeatures sélectionnées :", variables_selectionnees)


In [ ]:
X_selected = X[variables_selectionnees]
X_selected = sm.add_constant(X_selected, has_constant='add')


logit_model_Y = sm.Logit(Y, X_selected).fit(disp=0)


print(logit_model_Y.summary())


In [ ]:
Y_pred = logit_model_Y.predict(X_selected)

residus_Y = Y - Y_pred

### Régression Post-Lasso sur $D$ (traitement)

De façon symétrique, on applique le même procédé à la variable de traitement $D$ (1 si bourse, 0 sinon). Cette étape est cruciale : elle permet d'extraire la variation exogène dans l'assignation de la bourse, conditionnellement aux covariables.

Les résidus $\tilde{D} = D - \hat{m}(X)$ représentent la partie de la bourse qui n'est **pas** expliquée par les caractéristiques observables de l'étudiant. C'est précisément sur cette variation résiduelle que repose l'identification de l'effet causal.

Si $\hat{m}(X) = P(D=1|X)$ est bien estimé (hypothèse d'overlap satisfaite), les résidus $\tilde{D}$ sont approximativement indépendants des confounders $X$.


In [ ]:
lasso_D = LassoCV(cv=5, random_state=42).fit(X, D)

lasso_D_coef = pd.Series(lasso_D.coef_, index=X.columns)
print("Coefficients Lasso :\n", lasso_D_coef)

variables_selectionnees = lasso_D_coef[lasso_D_coef != 0].index.tolist()
print("\nFeatures sélectionnées :", variables_selectionnees)

In [ ]:
X_selected = X[variables_selectionnees]
X_selected = sm.add_constant(X_selected, has_constant='add')


logit_model_D = sm.Logit(D, X_selected).fit(disp=0)


print(logit_model_D.summary())


In [ ]:
D_pred = logit_model_D.predict(X_selected)

residus_D = D - D_pred

### Régression des résidus : identification de l'ATE

La troisième étape est la régression OLS de $\tilde{Y}$ sur $\tilde{D}$ :
$$\hat{\theta} = \frac{\sum_i \tilde{D}_i \tilde{Y}_i}{\sum_i \tilde{D}_i^2}$$

C'est l'application directe du **Théorème de Frisch-Waugh-Lovell** : le coefficient de $D$ dans la régression de $Y$ sur $D$ et $X$ est identique au coefficient de $\tilde{D}$ dans la régression de $\tilde{Y}$ sur $\tilde{D}$, après avoir partialisé $X$.

On utilise des **erreurs standard robustes HC3** (heteroskedasticity-consistent) pour tenir compte du fait que les résidus de la régression finale ne sont pas i.i.d. - ils héritent de l'incertitude des étapes d'estimation des nuisances.


In [ ]:
# Régression des résidus de Y sur les résidus de D
post_lasso_model = sm.OLS(residus_Y, sm.add_constant(residus_D)).fit(cov_type='HC3')

print(post_lasso_model.summary())

### Interprétation du Double Post-Lasso

Le Double Post-Lasso est une méthode en deux étapes :
1. On régresse l'outcome $Y$ (_dropout_ binaire) et le traitement $D$ (bourse) sur les covariables $X$ via un Lasso pour sélectionner les variables pertinentes
2. On régresse ensuite les résidus de $Y$ sur ceux de $D$ : le coefficient obtenu est notre estimateur de l'effet causal $\theta$

Cette méthode, introduite par **Belloni, Chernozhukov & Hansen (2014)**, permet de traiter le problème de sélection de variables dans les modèles à haute dimension tout en conservant des propriétés inférentielles valides.

#### Conditions pour la validité de l'estimateur

Deux hypothèses clés sont nécessaires :
- **Sparsité** : parmi toutes les covariables, seul un petit nombre $s \ll n$ contribue vraiment à la prédiction de $Y$ et $D$. Le Lasso est conçu pour exploiter cette structure.
- **Overlap** (positivité) : $0 < P(D=1|X) < 1$ pour tout $X$ - chaque étudiant doit avoir une chance positive d'être boursier ou non-boursier, conditionnellement à ses covariables.

Le coefficient ci-dessous représente l'**ATE** (Average Treatment Effect) de la bourse sur le risque de dropout, après avoir contrôlé l'ensemble des confounders observés.


In [ ]:
# Interprétation du résultat Post-Lasso

coef_dpl = post_lasso_model.params.iloc[1]
se_dpl   = post_lasso_model.bse.iloc[1]
pval_dpl = post_lasso_model.pvalues.iloc[1]
ci_low   = coef_dpl - 1.96 * se_dpl
ci_high  = coef_dpl + 1.96 * se_dpl
sig      = '***' if pval_dpl < 0.001 else ('**' if pval_dpl < 0.01 else ('*' if pval_dpl < 0.05 else 'n.s.'))

print(f"Double Post-Lasso - Effet causal de la bourse sur le Dropout")
print(f"θ = {coef_dpl:+.4f}  |  SE = {se_dpl:.4f}  |  IC 95% = [{ci_low:+.4f}, {ci_high:+.4f}]  |  p = {pval_dpl:.4f} {sig}")


### Visualisation - Forest plot provisoire (Double Post-Lasso)

On représente graphiquement l'estimé et son intervalle de confiance à 95 %.

In [ ]:
ACCENT_COLOR = 'blue'

fig, ax = plt.subplots(figsize=(10, 3.5))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

ax.barh([0], [coef_dpl],
        xerr=[[coef_dpl - ci_low], [ci_high - coef_dpl]],
        color=COLORS['dropout'], alpha=0.75, height=0.4,
        error_kw={'elinewidth': 2, 'capsize': 10, 'ecolor': TEXT_COLOR})

ax.axvline(0, color=ACCENT_COLOR, lw=1.8, ls='--', alpha=0.9,
           label='$\\theta = 0$ (aucun effet)')

ax.scatter([coef_dpl], [0], color=TEXT_COLOR, zorder=5, s=60)

ax.text(coef_dpl, -0.28, f'$\\theta$ = {coef_dpl:+.3f}\n{sig}',
        ha='center', va='top', fontsize=11, fontweight='bold', color=TEXT_COLOR)

ax.set_ylim(-0.7, 0.7)
# --------------------------

ax.set_yticks([0])
ax.set_yticklabels(['Double Post-Lasso\n(Dropout)'], fontsize=12, fontfamily=FONT_BODY)
ax.set_xlabel('ATE estimé (effet sur la probabilité de dropout)', fontsize=12, labelpad=10)

ax.set_title('Effet causal de la bourse scolaire sur le Dropout\n(Double Post-Lasso)',
             fontsize=15, fontfamily=FONT_TITLE, pad=20) # pad augmenté à 20

ax.legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR, loc='upper right')
ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Double Machine Learning (DML - Chernozhukov et al., 2018)

### Pourquoi aller au-delà du Double Post-Lasso ?

Le Double Post-Lasso repose sur l'hypothèse de **sparsité** : les fonctions de nuisance $\ell_0(X)$ et $m_0(X)$ doivent être bien approximées par des modèles linéaires creux. Cette hypothèse est forte - dans la réalité, les déterminants du dropout peuvent présenter des **non-linéarités** et des **interactions** que le Lasso ne capture pas.

Le **Double Machine Learning** (Chernozhukov, Chetverikov, Demirer, Duflo, Hansen, Newey & Robins, 2018, *Econometrics Journal*) généralise cette approche en remplaçant le Lasso par **n'importe quel algorithme de machine learning** pour estimer les nuisances, tout en conservant la validité de l'inférence.

### Le Partially Linear Model (PLM)

On continue de travailler sous le PLM :
$$Y = \theta_0 D + \ell_0(X) + \varepsilon, \quad \mathbb{E}[\varepsilon | D, X] = 0$$
$$D = m_0(X) + V, \quad \mathbb{E}[V | X] = 0$$

Les fonctions $\ell_0(X) = \mathbb{E}[Y|X]$ et $m_0(X) = \mathbb{E}[D|X]$ sont des **paramètres de nuisance** - on s'y intéresse uniquement pour identifier $\theta_0$, pas pour leur propre valeur.

### Deux problèmes sans cross-fitting

Si on estimait les nuisances et l'effet causal sur le **même échantillon**, on ferait face à deux biais :
1. **Biais de régularisation** : les algorithmes ML (Lasso, RF, GB) introduisent un biais pour réduire la variance. Ce biais se transmet à $\hat{\theta}$ si les nuisances sont estimées sur le même échantillon.
2. **Overfitting** : le modèle aurait mémorisé les données d'entraînement, faussant les résidus utilisés pour identifier $\theta_0$.

### Solution : le Cross-Fitting

Le cross-fitting (aussi appelé *sample splitting*) résout ces deux problèmes simultanément. L'échantillon est partitionné en $K$ folds. Pour chaque fold $k$ :
- Les nuisances $\hat{\ell}^{[k]}$ et $\hat{m}^{[k]}$ sont estimées sur les **folds $\neq k$**
- Les résidus $\tilde{Y}_i = Y_i - \hat{\ell}^{[k]}(X_i)$ et $\tilde{D}_i = D_i - \hat{m}^{[k]}(X_i)$ sont calculés pour $i \in$ fold $k$

L'estimateur final est :
$$\hat{\theta} = \left(\frac{1}{n}\sum_i \tilde{D}_i^2\right)^{-1} \frac{1}{n}\sum_i \tilde{D}_i \tilde{Y}_i$$

### Propriété centrale : Neyman-orthogonalité

Le score de moment $\psi_i = \tilde{D}_i(\tilde{Y}_i - \theta \tilde{D}_i)$ est **Neyman-orthogonal** par rapport aux nuisances : sa dérivée par rapport à $\ell_0$ et $m_0$ est nulle à leur vraie valeur. Cela implique que les erreurs dans l'estimation des nuisances n'affectent $\hat{\theta}$ qu'au **second ordre** - on peut donc utiliser des estimateurs ML convergents à vitesse $n^{-1/4}$ et obtenir quand même $\sqrt{n}$-normalité pour $\hat{\theta}$.

Formellement (Théorème 3.1 de Chernozhukov et al., 2018) :
$$\sqrt{n}(\hat{\theta} - \theta_0) \xrightarrow{d} \mathcal{N}(0, V), \quad V = \mathbb{E}[\tilde{D}^2]^{-2} \mathbb{E}[\tilde{D}^2 \varepsilon^2]$$

### Trois learners comparés

On compare trois classes d'estimateurs pour les nuisances :
- **Lasso** : paramétrique, sparse, rapide - hypothèse de linéarité
- **Random Forest** : non-paramétrique, gère bien les interactions et non-linéarités, variance réduite par bagging 
- **Gradient Boosting** : non-paramétrique, faible biais, mais potentiellement plus sujet à l'overfitting sans bonne régularisation 

La convergence des estimés entre learners est un test de **robustesse** : si $\hat{\theta}$ est stable, cela suggère que le résultat ne dépend pas de la forme fonctionnelle supposée pour les nuisances.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

try:
    import doubleml as dml
    from doubleml import DoubleMLData, DoubleMLPLR, DoubleMLIRM
    print("doubleml importé avec succès")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'doubleml', '-q'])
    import doubleml as dml
    from doubleml import DoubleMLData, DoubleMLPLR, DoubleMLIRM

from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=697)
X_raw = dataset.data.features
y_raw = dataset.data.targets
df_dml = pd.concat([X_raw, y_raw], axis=1)

D_dml = df_dml['Scholarship holder'].astype(int)
df_dml['Y_dropout']  = (df_dml['Target'] == 'Dropout').astype(int)
df_dml['Y_graduate'] = (df_dml['Target'] == 'Graduate').astype(int)
df_dml['Y_ordinal']  = df_dml['Target'].map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})

COVARIATES_BASE = [
    'Marital Status', 'Gender', 'Age at enrollment', 'Nacionality', 'International', 'Displaced',
    'Previous qualification', 'Previous qualification (grade)',
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
    'Admission grade', 'Application mode', 'Application order',
    'Course', 'Daytime/evening attendance',
    'Debtor', 'Tuition fees up to date',
    'Educational special needs',
    'Unemployment rate', 'Inflation rate', 'GDP',
]

X_dml = df_dml[COVARIATES_BASE].copy()

print(f"Dataset : {len(df_dml)} étudiants | Boursiers : {D_dml.sum()} ({D_dml.mean()*100:.1f}%)")
print(f"Covariables pré-traitement : {len(COVARIATES_BASE)}")


### Construction des objets `DoubleMLData` et estimation PLR

On construit les objets `DoubleMLData` pour les deux outcomes (dropout et graduate), puis on entraîne le modèle **PLR** (*Partially Linear Regression*) avec les trois learners.

Le package `DoubleML` (Bach et al., 2022) implémente fidèlement la procédure de Chernozhukov et al. (2018) : pour chaque combinaison learner × outcome, il effectue automatiquement le **cross-fitting à 5 folds**, répété 2 fois (`n_rep=2`) pour réduire la variance due au découpage aléatoire. L'estimateur final est la médiane des estimés sur les répétitions.

**Paramétrage des learners :**
- Le Lasso utilise `LassoCV` avec standardisation préalable (`StandardScaler`) - indispensable car le Lasso est sensible à l'échelle des variables.
- Le Random Forest est limité à `max_depth=6` et `min_samples_leaf=10` pour éviter l'overfitting, conformément aux recommandations de Wager & Athey (2018) pour les RF en inférence causale.
- Le Gradient Boosting utilise un faible learning rate (`0.05`) pour régulariser l'entraînement.


In [ ]:
# ─── Objets DoubleMLData ────────────────────────────────────
dml_data_dropout = dml.DoubleMLData(
    pd.concat([X_dml.reset_index(drop=True),
               D_dml.reset_index(drop=True).rename('Scholarship holder'),
               df_dml['Y_dropout'].reset_index(drop=True)], axis=1),
    y_col='Y_dropout', d_cols='Scholarship holder', x_cols=COVARIATES_BASE
)
dml_data_graduate = dml.DoubleMLData(
    pd.concat([X_dml.reset_index(drop=True),
               D_dml.reset_index(drop=True).rename('Scholarship holder'),
               df_dml['Y_graduate'].reset_index(drop=True)], axis=1),
    y_col='Y_graduate', d_cols='Scholarship holder', x_cols=COVARIATES_BASE
)

# ─── Learners candidats ─────────────────────────────────────
learners = {
    'Lasso': {
        'ml_l': Pipeline([('sc', StandardScaler()), ('m', LassoCV(cv=5, max_iter=2000))]),
        'ml_m': Pipeline([('sc', StandardScaler()), ('m', LogisticRegressionCV(cv=5, max_iter=1000, penalty='l1', solver='saga'))])
    },
    'Random Forest': {
        'ml_l': RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1),
        'ml_m': RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)
    },
    'Gradient Boosting': {
        'ml_l': GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
        'ml_m': GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
    }
}

results_dml = []
for learner_name, lrn in learners.items():
    for outcome_name, dml_data in [('Dropout', dml_data_dropout), ('Graduate', dml_data_graduate)]:
        try:
            model = dml.DoubleMLPLR(
                dml_data, ml_l=lrn['ml_l'], ml_m=lrn['ml_m'],
                n_folds=5, n_rep=2, score='partialling out'
            )
            model.fit()
            coef = model.coef[0]; se = model.se[0]; pval = model.pval[0]
            ci   = model.confint(level=0.95)
            results_dml.append({
                'Learner': learner_name,
                'Outcome': outcome_name,
                'θ (ATE)': round(coef, 4),
                'SE': round(se, 4),
                'IC95 low': round(ci.iloc[0, 0], 4),
                'IC95 high': round(ci.iloc[0, 1], 4),
                'p-value': round(pval, 4),
                'Sig.': '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
            })
            print(f"✔ DML-PLR [{learner_name:>20s}] [{outcome_name}]  θ={coef:+.4f}  SE={se:.4f}  p={pval:.4f}")
        except Exception as e:
            print(f"✗ [{learner_name}][{outcome_name}] Erreur : {e}")

results_dml_df = pd.DataFrame(results_dml)
print("\n" + "═"*70)
print("TABLEAU COMPARATIF DML - PLR")
print("═"*70)
print(results_dml_df.to_string(index=False))


### Sélection de modèle et validation croisée

La sélection du meilleur learner est implicitement réalisée par le DML via la validation croisée à 5 plis intégrée dans chaque estimateur. La comparaison des ATE obtenus avec différents learners permet de vérifier la **robustesse** de l'estimation : si les trois learners convergent vers des estimés proches, cela renforce la crédibilité du résultat.

Ce principe de robustesse est central dans la littérature DML. Comme le soulignent Chernozhukov et al. (2018), la Neyman-orthogonalité garantit que **des erreurs de premier ordre dans les nuisances ne biaisent pas $\hat{\theta}$**, mais cette garantie est asymptotique. En pratique, un learner très mal spécifié (ex : arbre de décision profond sans régularisation) pourrait introduire un biais fini. La convergence des estimés entre Lasso, Random Forest et Gradient Boosting constitue donc un **test informel de spécification**.

On visualise ci-dessous les estimés et intervalles de confiance pour chaque combinaison learner × outcome.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Comparaison des estimateurs DML-PLR par learner',
             fontsize=16, fontfamily=FONT_TITLE, y=1.01)

col_theta = [c for c in results_dml_df.columns if 'ATE' in c][0]

for ax, outcome, color in zip(axes,
                               ['Dropout', 'Graduate'],
                               [COLORS['dropout'], COLORS['graduate']]):
    ax.set_facecolor(BG_COLOR)
    sub = results_dml_df[results_dml_df['Outcome'] == outcome].reset_index(drop=True)
    y_pos = list(range(len(sub)))
    ax.barh(y_pos, sub[col_theta],
            xerr=[sub[col_theta] - sub['IC95 low'], sub['IC95 high'] - sub[col_theta]],
            color=color, alpha=0.75, height=0.45,
            error_kw={'elinewidth': 1.8, 'capsize': 6, 'ecolor': TEXT_COLOR},
            edgecolor=BG_COLOR)
    ax.scatter(sub[col_theta], y_pos, color=TEXT_COLOR, zorder=5, s=50)
    ax.axvline(0, color=ACCENT_COLOR, lw=1.8, ls='--', alpha=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['Learner'], fontsize=11, fontfamily=FONT_BODY)
    ax.set_title(f'Outcome : {outcome}', fontsize=14, fontfamily=FONT_TITLE)
    ax.set_xlabel('ATE estimé', fontsize=12, labelpad=8)
    ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
    ax.set_axisbelow(True)
    for i, row in sub.iterrows():
        ax.text(row[col_theta], i + 0.30,
                f"θ={row[col_theta]:+.3f} {row['Sig.']}",
                ha='center', fontsize=9, color=TEXT_COLOR)

plt.tight_layout()
plt.show()


### DML-IRM : Interactive Regression Model

Le DML-PLR impose une contrainte forte : l'effet du traitement $D$ sur $Y$ est **additif et constant** (pas d'interaction $D \times X$). C'est l'hypothèse de **traitement homogène**. Dans notre contexte, il est raisonnable de penser que l'effet de la bourse varie selon le profil de l'étudiant,c'est ce qu'on appelle l'**hétérogénéité de l'effet de traitement !**

#### Le modèle IRM

L'**Interactive Regression Model** (Chernozhukov et al., 2018) généralise le PLR :
$$Y = g_0(D, X) + \varepsilon, \quad \mathbb{E}[\varepsilon | X, D] = 0$$
$$D = m_0(X) + \tilde{\varepsilon}, \quad \mathbb{E}[\tilde{\varepsilon} | X] = 0, \quad D \in \{0, 1\}$$

où $g_0(D, X)$ est une fonction **totalement libre** des interactions entre $D$ et $X$. Le paramètre d'intérêt est l'**Average Predictive Effect (APE)**, qui, sous l'hypothèse d'ignorabilité conditionnelle $Y(0), Y(1) \perp D | X$, coïncide avec l'**ATE** :
$$\theta_0 = \mathbb{E}[g_0(1, X) - g_0(0, X)]$$

#### Estimateur et score orthogonal

Trois fonctions de nuisance sont nécessaires :
- $g_0(X) = \mathbb{E}[Y|D=0, X]$ et $g_1(X) = \mathbb{E}[Y|D=1, X]$ : modèles de résultat séparés pour traités et contrôles
- $m_0(X) = P(D=1|X)$ : propensity score

L'estimateur DML-IRM utilise le score orthogonal (**score AIPW**) :
$$\psi_i = \hat{g}_1(X_i) - \hat{g}_0(X_i) + \frac{D_i(Y_i - \hat{g}_1(X_i))}{\hat{m}(X_i)} - \frac{(1-D_i)(Y_i - \hat{g}_0(X_i))}{1-\hat{m}(X_i)}$$

Ce score est **doublement robuste** : il est consistant si *au moins un* des deux modèles ($\hat{g}$ ou $\hat{m}$) est correctement spécifié. On vérifie également que la condition d'**overlap** est respectée en imposant un trimming : `trimming_threshold=0.01` élimine les observations pour lesquelles $\hat{m}(X) < 0.01$ ou $\hat{m}(X) > 0.99$.

#### IRM vs PLR : quand préférer l'un à l'autre ?

Comme le montrent les slides du cours, si le vrai modèle est l'IRM mais qu'on applique le PLR, on estime un **weighted ATE** avec des poids d'*overlap* $m(X)(1-m(X))$ - donnant plus de poids aux observations à propensity score proche de 0.5. Si l'effet est homogène ($g_1(X) - g_0(X) = \beta$ constant), les deux estimateurs coïncident. Dans le cas contraire, l'IRM est plus flexible et estime l'ATE non pondéré.


In [ ]:
# DML-IRM (ATE) avec Random Forest

irm_results = {}
for outcome_name, dml_data in [('Dropout', dml_data_dropout), ('Graduate', dml_data_graduate)]:
    try:
        irm = dml.DoubleMLIRM(
            dml_data,
            ml_g=RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=10,
                                       random_state=42, n_jobs=-1),
            ml_m=RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=10,
                                        random_state=42, n_jobs=-1),
            n_folds=5, n_rep=2, score='ATE', trimming_threshold=0.01
        )
        irm.fit()
        coef = irm.coef[0]; se = irm.se[0]; pval = irm.pval[0]
        ci   = irm.confint(level=0.95)
        irm_results[outcome_name] = {'coef': coef, 'se': se, 'pval': pval, 'ci': ci}
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.'))
        print(f"DML-IRM [{outcome_name}]  ATE = {coef:+.4f}  SE = {se:.4f}  "
              f"IC95 = [{ci.iloc[0,0]:+.4f}, {ci.iloc[0,1]:+.4f}]  p = {pval:.4f} {sig}")
    except Exception as e:
        print(f"Erreur IRM [{outcome_name}] : {e}")


## Estimation par Score de Propension et AIPW

### Propensity Score et vérification de l'overlap

Avant d'utiliser des estimateurs basés sur le propensity score, il est essentiel de vérifier l'**hypothèse d'overlap** (ou *common support*) : pour chaque valeur des covariables, tout étudiant doit avoir une probabilité strictement positive d'être boursier **et** non-boursier. Formellement :
$$\exists\, \varepsilon > 0 : \varepsilon < P(D=1|X) < 1-\varepsilon \quad \text{p.s.}$$

Cette condition est nécessaire pour identifier l'ATE. En pratique, on la vérifie en inspectant le **chevauchement des distributions** du propensity score entre traités et contrôles. Si les distributions ne se chevauchent pas, les observations dans les zones sans chevauchement ne peuvent pas servir à l'identification causale.

On estime $e(X) = P(D=1|X)$ par **régression logistique régularisée** (pénalité L1, validation croisée stratifiée à 5 plis). La pénalité L1 est préférable à L2 ici car elle produit des modèles creux - cohérent avec la structure sparse probable du propensity score.


In [ ]:
cv_ps = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ps_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(cv=5, max_iter=1000, random_state=42, penalty='l1', solver='saga'))
])
ps_scores = cross_val_predict(ps_model, X_dml, D_dml, cv=cv_ps, method='predict_proba')[:, 1]
df_dml['propensity_score'] = ps_scores
auc_ps = roc_auc_score(D_dml, ps_scores)
print(f"AUC du propensity score : {auc_ps:.3f}")
print(f"PS Boursiers    : min={ps_scores[D_dml==1].min():.3f}  max={ps_scores[D_dml==1].max():.3f}  moy={ps_scores[D_dml==1].mean():.3f}")
print(f"PS Non-boursiers: min={ps_scores[D_dml==0].min():.3f}  max={ps_scores[D_dml==0].max():.3f}  moy={ps_scores[D_dml==0].mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for _ax in axes: _ax.set_facecolor(BG_COLOR)
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle("Propensity Score - Qualite et Chevauchement",
             fontsize=16, fontweight='bold', fontfamily=FONT_TITLE)

for val, label, color in [(0,'Non-Boursier',COLORS['control']),(1,'Boursier',COLORS['treated'])]:
    sns.kdeplot(df_dml.loc[D_dml==val,'propensity_score'], ax=axes[0],
                fill=True, alpha=0.45, color=color, label=label)
axes[0].axvline(0.1, ls='--', color=ACCENT_COLOR, lw=1.5, label='Seuil trimming')
axes[0].axvline(0.9, ls='--', color=ACCENT_COLOR, lw=1.5)
axes[0].set_title(f"Densites du Propensity Score (AUC = {auc_ps:.3f})",
                  fontsize=13, fontfamily=FONT_TITLE)
axes[0].set_xlabel("P(Bourse = 1 | X)", fontsize=12)
axes[0].legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)

bins = np.linspace(0, 1, 30)
axes[1].hist(df_dml.loc[D_dml==1,'propensity_score'], bins=bins,
             color=COLORS['treated'], alpha=0.75, label='Boursier', density=True, edgecolor=BG_COLOR)
axes[1].hist(df_dml.loc[D_dml==0,'propensity_score'], bins=bins,
             color=COLORS['control'], alpha=0.75, label='Non-Boursier', density=True, edgecolor=BG_COLOR)
axes[1].set_title("Histogramme du Propensity Score", fontsize=13, fontfamily=FONT_TITLE)
axes[1].set_xlabel("P(Bourse = 1 | X)", fontsize=12)
axes[1].legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
plt.tight_layout(); plt.show()
print("Chevauchement verifie. AUC moderee = bonne separabilite sans assignation deterministe.")


### Estimateur AIPW (Augmented Inverse Propensity Weighting)

L'estimateur **AIPW** (*Augmented IPW*), aussi appelé **doublement robuste** (*doubly robust*), est l'un des estimateurs semi-paramétriques les plus efficaces pour l'ATE (Robins, Rotnitzky & Zhao, 1994 ; Scharfstein, Rotnitzky & Robins, 1999).

Il combine deux modèles :
- Un modèle de propensity score $\hat{e}(X) = P(D=1|X)$
- Un modèle de résultat (*outcome model*) $\hat{\mu}_d(X) = \mathbb{E}[Y|D=d, X]$

L'estimateur est :
$$\hat{\theta}_{AIPW} = \frac{1}{n} \sum_i \underbrace{\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i)}_{\text{partie DR}} + \underbrace{\frac{D_i(Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1-D_i)(Y_i - \hat{\mu}_0(X_i))}{1-\hat{e}(X_i)}}_{\text{correction IPW}}$$

#### Propriété de double robustesse

L'AIPW est **consistant** si *au moins un* des deux modèles est correctement spécifié :
- Si $\hat{\mu}_d$ est correct mais $\hat{e}$ ne l'est pas → les corrections IPW convergent vers 0 et l'estimateur est dominé par la partie outcome
- Si $\hat{e}$ est correct mais $\hat{\mu}_d$ ne l'est pas → les corrections IPW corrigent le biais résiduel

De plus, l'AIPW atteint la **borne de variance semi-paramétrique** (Hahn, 1998) : il est asymptotiquement efficient parmi tous les estimateurs réguliers de l'ATE.

#### Lien avec le DML-IRM

On remarquera que le score AIPW ci-dessus est **exactement** le score orthogonal $\psi_i$ du DML-IRM (cf. section précédente). Les deux approches sont donc équivalentes dans leur formulation - la différence réside dans le fait que l'implémentation AIPW ci-dessous nous donne accès aux **scores individuels $\psi_i$**, qui seront utilisés dans la section suivante pour estimer les effets hétérogènes.

Le cross-fitting (5 folds) est appliqué ici aussi : les modèles $\hat{\mu}_d$ et $\hat{e}$ sont estimés hors-échantillon pour éviter l'overfitting, conformément aux recommandations de Chernozhukov et al. (2018).


In [ ]:
def aipw_estimator(Y, D, X, outcome_name="Y"):
    """
    AIPW doublement robuste.
    Utilise des Random Forests cross-fités pour les deux modèles.
    """
    Y_arr = np.array(Y); D_arr = np.array(D)
    cv_aipw = KFold(n_splits=5, shuffle=True, random_state=42)
    X_arr   = np.array(X)

    # ── Propensity score cross-fité ──────────────────────────
    ps_est = Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegressionCV(cv=5, max_iter=1000, penalty='l1', solver='saga'))
    ])
    e_hat = cross_val_predict(ps_est, X_arr, D_arr, cv=5, method='predict_proba')[:, 1]
    e_hat = np.clip(e_hat, 0.02, 0.98)  # trimming pour la stabilité numérique

    # ── Outcome models cross-fités ───────────────────────────
    mu1_hat = np.zeros(len(Y_arr))
    mu0_hat = np.zeros(len(Y_arr))
    rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)
    scaler_cv = StandardScaler()
    X_sc = scaler_cv.fit_transform(X_arr)
    for train_idx, val_idx in cv_aipw.split(X_sc):
        rf.fit(X_sc[train_idx][D_arr[train_idx] == 1], Y_arr[train_idx][D_arr[train_idx] == 1])
        mu1_hat[val_idx] = rf.predict(X_sc[val_idx])
        rf.fit(X_sc[train_idx][D_arr[train_idx] == 0], Y_arr[train_idx][D_arr[train_idx] == 0])
        mu0_hat[val_idx] = rf.predict(X_sc[val_idx])

    # ── Score AIPW ────────────────────────────────────────────
    psi = (mu1_hat - mu0_hat
           + D_arr * (Y_arr - mu1_hat) / e_hat
           - (1 - D_arr) * (Y_arr - mu0_hat) / (1 - e_hat))

    theta = psi.mean()
    se    = psi.std() / np.sqrt(len(Y_arr))
    return {
        'method': 'AIPW',
        'outcome': outcome_name,
        'theta': theta,
        'se': se,
        'ci_low': theta - 1.96 * se,
        'ci_high': theta + 1.96 * se,
        'pval': 2 * (1 - stats.norm.cdf(abs(theta) / se)),
        'psi': psi,
    }

print("Estimation AIPW en cours (peut prendre quelques minutes)…")
aipw_dropout  = aipw_estimator(df_dml['Y_dropout'].values,  D_dml.values, X_dml, "Dropout")
aipw_graduate = aipw_estimator(df_dml['Y_graduate'].values, D_dml.values, X_dml, "Graduate")

for res in [aipw_dropout, aipw_graduate]:
    sig = '***' if res['pval'] < 0.001 else ('**' if res['pval'] < 0.01 else ('*' if res['pval'] < 0.05 else 'n.s.'))
    print(f"[AIPW {res['outcome']:>8s}]  θ = {res['theta']:+.4f}  SE = {res['se']:.4f}  IC95 = [{res['ci_low']:+.4f}, {res['ci_high']:+.4f}]  p = {res['pval']:.4f} {sig}")


Les scores AIPW individuels $\psi_i$ seront réutilisés dans la section suivante pour l'analyse des effets hétérogènes et le policy learning.

## Effets Hétérogènes de Traitement (HTE)

L'ATE moyen masque potentiellement une **hétérogénéité importante** dans l'effet de la bourse. La question est : *pour qui la bourse est-elle la plus efficace ?*

### Approche : scores AIPW par sous-groupe

Nous estimons les **CATE** (*Conditional Average Treatment Effects*) par sous-groupes en utilisant les scores AIPW individuels $\psi_i$ comme approximation du CATE individuel. Pour chaque sous-groupe $g$ :
$$\hat{\theta}_g = \frac{1}{|g|} \sum_{i \in g} \psi_i, \quad SE_g = \frac{\text{std}(\psi_i)_{i \in g}}{\sqrt{|g|}}$$

Cette approche repose sur la propriété que $\mathbb{E}[\psi_i | X_i \in g] = \theta_g$ (le CATE du groupe $g$), qui découle directement de la propriété $\mathbb{E}[\psi_i | X_i] = \tau(X_i)$ des scores doublement robustes (Chernozhukov, Demirer, Duflo & Fernandez-Val, 2020, *Journal of Applied Econometrics*).

Cette procédure, appelée **Generic Machine Learning** (GML) permet d'estimer des effets hétérogènes sans avoir à spécifier ex ante les groupes d'intérêt, tout en conservant une inférence valide.

#### Mise en garde (!)

Ces comparaisons par sous-groupe sont **exploratoires** : elles ne sont pas corrigées pour les tests multiples. Pour une analyse formelle de l'hétérogénéité, on utiliserait des méthodes comme le **Causal Forest** (Wager & Athey, 2018) ou le test de Best Linear Predictor du CATE (Chernozhukov et al., 2020).


In [ ]:
psi_dropout  = aipw_dropout['psi']
psi_graduate = aipw_graduate['psi']

subgroups = {
    'Genre'           : ('Gender',                   {0: 'Femme', 1: 'Homme'}),
    'Âge'             : ('Age at enrollment',         lambda x: '≤20 ans' if x <= 20 else ('21-25 ans' if x <= 25 else '>25 ans')),
    'Débiteur'        : ('Debtor',                   {0: 'Non-débiteur', 1: 'Débiteur'}),
    'Frais à jour'    : ('Tuition fees up to date',  {0: 'Non', 1: 'Oui'}),
    'Horaire'         : ('Daytime/evening attendance', {0: 'Cours du soir', 1: 'Cours de jour'}),
    'Qual. mère'      : ("Mother's qualification",   lambda x: 'Faible (≤9)' if x <= 9 else ('Moyen (10-19)' if x <= 19 else 'Élevé (≥20)')),
    'Genre bénéf.'    : ('Gender',                   {0: 'Femme', 1: 'Homme'}),
}

hte_rows = []
for sg_name, (col, mapper) in subgroups.items():
    if callable(mapper):
        groups = df_dml[col].apply(mapper)
    else:
        groups = df_dml[col].map(mapper).fillna('Autre')
    for g_val in groups.unique():
        mask = (groups == g_val).values
        if mask.sum() < 30:
            continue
        theta_d = psi_dropout[mask].mean()
        se_d    = psi_dropout[mask].std() / np.sqrt(mask.sum())
        theta_g = psi_graduate[mask].mean()
        se_g    = psi_graduate[mask].std() / np.sqrt(mask.sum())
        hte_rows.append({
            'Sous-groupe': sg_name,
            'Valeur': str(g_val),
            'N': mask.sum(),
            'CATE Dropout':  round(theta_d, 4),
            'SE Dropout':    round(se_d, 4),
            'CI_low_D':  round(theta_d - 1.96 * se_d, 4),
            'CI_high_D': round(theta_d + 1.96 * se_d, 4),
            'CATE Graduate': round(theta_g, 4),
            'SE Graduate':   round(se_g, 4),
        })

hte_df = pd.DataFrame(hte_rows).drop_duplicates(subset=['Sous-groupe', 'Valeur']).reset_index(drop=True)
print("EFFETS HÉTÉROGÈNES PAR SOUS-GROUPE :")
print(hte_df[['Sous-groupe', 'Valeur', 'N', 'CATE Dropout', 'SE Dropout', 'CATE Graduate']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Effets Hétérogènes de la Bourse par Sous-groupe (AIPW Scores)',
             fontsize=16, fontfamily=FONT_TITLE, y=1.01)

for ax, outcome_col, se_col, title, base_color, direction in [
    (axes[0],'CATE Dropout','SE Dropout','CATE sur Dropout (négatif = protecteur)',COLORS['dropout'],'neg'),
    (axes[1],'CATE Graduate','SE Graduate','CATE sur Graduate (positif = favorable)',COLORS['graduate'],'pos'),
]:
    ax.set_facecolor(BG_COLOR)
    labels = [f"{r['Sous-groupe']} : {r['Valeur']}\n(N={r['N']})" for _, r in hte_df.iterrows()]
    vals   = hte_df[outcome_col].values
    ses    = hte_df[se_col].values
    colors = [base_color if (v < 0 if direction=='neg' else v > 0) else COLORS['neutral'] for v in vals]
    y_pos  = list(range(len(vals)))
    ax.barh(y_pos, vals, xerr=1.96*ses, color=colors, alpha=0.75,
            error_kw={'elinewidth': 1.5, 'capsize': 4, 'ecolor': TEXT_COLOR},
            edgecolor=BG_COLOR)
    ax.scatter(vals, y_pos, color=TEXT_COLOR, zorder=5, s=35)
    ax.axvline(0, color=ACCENT_COLOR, lw=1.8, ls='--', alpha=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9, fontfamily=FONT_BODY)
    ax.set_xlabel('CATE (Effet Causal Moyen Conditionnel)', fontsize=12, labelpad=8)
    ax.set_title(title, fontsize=14, fontfamily=FONT_TITLE)
    ax.grid(axis='x', color=GRID_COLOR, alpha=0.5)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


**Lecture des graphiques HTE :** Chaque barre représente le CATE estimé pour un sous-groupe. Les barres d'erreur sont les intervalles de confiance à 95 %. Une barre rouge (à gauche de 0) pour le dropout signifie que la bourse réduit le risque de décrochage pour ce groupe ; une barre verte (à droite de 0) pour le graduate signifie que la bourse augmente les chances de diplomation.

Les résultats permettent d'identifier quels profils d'étudiants bénéficient le plus de la bourse, information précieuse pour orienter la politique d'attribution.

## Policy Learning : Allocation Optimale des Bourses

La *policy learning* vise à répondre à la question : **quelle règle d'attribution des bourses maximise le bien-être collectif ?** Plutôt que d'attribuer les bourses selon les critères actuels (essentiellement sociaux), on cherche à identifier les étudiants pour qui l'effet causal de la bourse est le plus fort.

### Cadre formel

Une politique est une fonction $\pi : \mathcal{X} \to \{0, 1\}$ qui, à chaque profil de covariables $X$, associe une décision d'attribution ($1$) ou non ($0$). La **valeur** d'une politique est :
$$V(\pi) = \mathbb{E}[\pi(X_i) \cdot \tau(X_i)]$$
où $\tau(X_i) = \mathbb{E}[Y(1) - Y(0) | X_i]$ est le CATE individuel.

La politique optimale est $\pi^*(X) = \mathbf{1}[\tau(X) < 0]$ (attribuer la bourse si elle réduit le dropout). Comme $\tau(X_i)$ n'est pas directement observable, on l'approche par les scores AIPW $\psi_i$ et on entraîne un Random Forest pour prédire ces scores - c'est l'approche de **Kitagawa & Tetenov (2018)** et **Athey & Wager (2021)**.

### Évaluation de la politique

La valeur des différentes politiques est estimée via les scores AIPW : $\hat{V}(\pi) = \frac{1}{n}\sum_i \pi(X_i) \psi_i$. Cette estimateur est non biaisé grâce à la double robustesse des scores $\psi_i$.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

scaler_pl = StandardScaler()
X_scaled_pl = scaler_pl.fit_transform(X_dml)

rf_cate = RandomForestRegressor(
    n_estimators=300, max_depth=5, min_samples_leaf=15, random_state=42, n_jobs=-1
)
rf_cate.fit(X_scaled_pl, psi_dropout)
cate_pred = rf_cate.predict(X_scaled_pl)

df_dml['cate_dropout']   = cate_pred
df_dml['policy_optimal'] = (cate_pred < np.percentile(cate_pred, 75)).astype(int)  # 25% les plus à risque
df_dml['policy_current'] = D_dml.values

def policy_value(policy_arr, psi_scores):
    """Valeur d'une politique via les scores AIPW."""
    return np.mean(policy_arr * psi_scores)

pv_current = policy_value(df_dml['policy_current'].values, psi_dropout)
pv_optimal = policy_value(df_dml['policy_optimal'].values, psi_dropout)
pv_all     = np.mean(psi_dropout)
pv_none    = 0.0

print("VALEURS DES POLITIQUES (réduction de décrochage attendue) :")
print(f"  Politique actuelle (observée)    : {pv_current:+.4f}")
print(f"  Politique optimale (CATE-based)  : {pv_optimal:+.4f}")
print(f"  Politique \"tout le monde\"       : {pv_all:+.4f}")
print(f"  Politique \"personne\"            : {pv_none:+.4f}")
print()
print("Interprétation :")
print(f"  Un score plus négatif = plus forte réduction du décrochage attendue.")
if pv_optimal < pv_current:
    print(f"  La politique optimale améliore de {abs(pv_optimal - pv_current):.4f} points la valeur par rapport à la politique actuelle.")


In [ ]:
feat_imp = pd.DataFrame({
    'Variable': X_dml.columns,
    'Importance': rf_cate.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Policy Learning - CATE individuels et Variables déterminantes',
             fontsize=16, fontfamily=FONT_TITLE, y=1.01)

# Distribution des CATE
axes[0].set_facecolor(BG_COLOR)
axes[0].hist(cate_pred, bins=40, color=COLORS['effect'], alpha=0.7,
             edgecolor=BG_COLOR, linewidth=0.5)
axes[0].axvline(0, color=ACCENT_COLOR, lw=2, ls='--', alpha=0.9,
                label='$\\theta = 0$ (indifférence)')
axes[0].axvline(np.percentile(cate_pred, 25), color=COLORS['treated'],
                lw=1.8, ls='--', alpha=0.8,
                label=f"25e pctile = {np.percentile(cate_pred, 25):.3f}")
axes[0].set_title('Distribution des CATE individuels (Dropout)',
                  fontsize=14, fontfamily=FONT_TITLE)
axes[0].set_xlabel('CATE estimé (impact sur le dropout)', fontsize=12, labelpad=8)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].legend(fontsize=10, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
axes[0].grid(color=GRID_COLOR, alpha=0.4)
axes[0].set_axisbelow(True)

# Feature importances
axes[1].set_facecolor(BG_COLOR)
axes[1].barh(feat_imp['Variable'][::-1], feat_imp['Importance'][::-1],
             color=COLORS['effect'], alpha=0.75, edgecolor=BG_COLOR)
axes[1].set_title('Variables clés pour la politique optimale',
                  fontsize=14, fontfamily=FONT_TITLE)
axes[1].set_xlabel('Importance (RF CATE)', fontsize=12, labelpad=8)
axes[1].grid(axis='x', color=GRID_COLOR, alpha=0.4)
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()
print('Distribution des CATE : majorité des effets négatifs = bourse protectrice en moyenne.')


## Analyse de Sensibilité et Tests Placebo

Pour valider la robustesse de nos résultats, nous effectuons deux types de vérifications classiques en économétrie causale.

### 1. Test placebo (permutation)

On permute aléatoirement l'assignation des bourses et on recalcule l'estimateur AIPW - répété 500 fois. Ce test simule la distribution nulle $H_0 : \theta = 0$ (aucun effet causal).

Si notre effet estimé $\hat{\theta}$ est bien causal, il devrait tomber dans les queues de la distribution sous permutation. La **p-valeur de permutation** $p = P(|\hat{\theta}_{H_0}| \geq |\hat{\theta}|)$ est un test non-paramétrique valide qui ne repose sur aucune hypothèse distributionnelle (Fisher, 1935).

### 2. Robustesse à la spécification des covariables

On répète l'estimation AIPW avec différents ensembles de covariables - de la spécification minimale (démographie seule) à la spécification complète (avec variables macro). Cette analyse teste si le résultat est stable ou dépend d'un choix arbitraire de contrôles.

Une mise en garde importante concerne les **variables post-traitement** (notes du premier semestre, crédits obtenus...). Les inclure comme covariables **ferme des chemins causaux médiateurs** (D → Médiateur → Y), ce qui biaise $\hat{\theta}$ vers zéro. Notre DAG identifie explicitement ces variables comme post-traitement, justifiant leur exclusion du modèle principal.


In [ ]:
# ─── Test placebo ───────────────────────────────────────────
np.random.seed(42)
n_placebo = 1000
placebo_thetas = []

for _ in range(n_placebo):
    D_fake = np.random.permutation(D_dml.values)
    e_pl   = D_fake.mean() * np.ones(len(D_fake))
    mu1_pl = df_dml['Y_dropout'].values[D_fake == 1].mean()
    mu0_pl = df_dml['Y_dropout'].values[D_fake == 0].mean()
    psi_pl = (mu1_pl - mu0_pl
              + D_fake * (df_dml['Y_dropout'].values - mu1_pl) / (e_pl + 1e-6)
              - (1 - D_fake) * (df_dml['Y_dropout'].values - mu0_pl) / (1 - e_pl + 1e-6))
    placebo_thetas.append(psi_pl.mean())

placebo_thetas = np.array(placebo_thetas)
true_theta     = aipw_dropout['theta']
p_placebo      = np.mean(np.abs(placebo_thetas) >= np.abs(true_theta))

print(f"Test Placebo ({n_placebo} permutations)")
print(f"  Estimé réel     : θ = {true_theta:+.4f}")
print(f"  Distribution H0 : μ = {placebo_thetas.mean():.4f}  σ = {placebo_thetas.std():.4f}")
print(f"  p-valeur placebo : {p_placebo:.4f}")
print(f"  → {'Effet significatif (H0 rejetée)' if p_placebo < 0.05 else 'Effet non distinguable du hasard'}")


In [ ]:
# ─── Robustesse aux ensembles de covariables ────────────────
covariate_sets = {
    'Démographie seule':   ['Gender', 'Age at enrollment', 'Marital Status', 'Displaced'],
    'Démographie + Académique': ['Gender', 'Age at enrollment', 'Admission grade',
                                  'Previous qualification', 'Previous qualification (grade)'],
    'Complet sans macro':  [c for c in COVARIATES_BASE if c not in ['Unemployment rate', 'Inflation rate', 'GDP']],
    'Complet avec macro':  COVARIATES_BASE,
}

POST_TREATMENT_COLS = [
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)',
]
covariate_sets['Complet + Post-traitement'] = COVARIATES_BASE + POST_TREATMENT_COLS

sensitivity_rows = []
for set_name, cols in covariate_sets.items():
    X_sens = df_dml[cols].copy()
    res = aipw_estimator(df_dml['Y_dropout'].values, D_dml.values, X_sens, set_name)
    sensitivity_rows.append({
        'Ensemble de covariables': set_name,
        'N covariables': len(cols),
        'θ (ATE)': round(res['theta'], 4),
        'SE': round(res['se'], 4),
        'IC95 low': round(res['ci_low'], 4),
        'IC95 high': round(res['ci_high'], 4),
        'p-value': round(res['pval'], 4),
        'Sig.': '***' if res['pval'] < 0.001 else ('**' if res['pval'] < 0.01 else ('*' if res['pval'] < 0.05 else 'n.s.'))
    })
    print(f"  {set_name:40s} θ={res['theta']:+.4f}  SE={res['se']:.4f}  p={res['pval']:.4f}")

sensitivity_df = pd.DataFrame(sensitivity_rows)
print("\nTABLEAU DE SENSIBILITÉ :")
print(sensitivity_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Analyse de Sensibilité - Tests Placebo et Robustesse aux Covariables',
             fontsize=16, fontfamily=FONT_TITLE, y=1.01)

# ── Test placebo ─────────────────────────────────────────────
axes[0].set_facecolor(BG_COLOR)
axes[0].hist(placebo_thetas, bins=40, color=COLORS['placebo'], alpha=0.75,
             label='Distribution placebo (H$_0$)', edgecolor=BG_COLOR)
axes[0].axvline(true_theta, color=ACCENT_COLOR, lw=2.5, ls='-',
                label=f'θ réel = {true_theta:.4f}')
for sign in [-1, 1]:
    axes[0].axvline(sign * np.percentile(np.abs(placebo_thetas), 95),
                    color=COLORS['effect'], ls='--', lw=1.5,
                    label='Seuil 5% bilatéral' if sign == 1 else '')
axes[0].set_title(f'Test Placebo ({n_placebo} permutations) - p = {p_placebo:.4f}',
                  fontsize=14, fontfamily=FONT_TITLE)
axes[0].set_xlabel('θ sous H$_0$ (traitement aléatoire)', fontsize=12, labelpad=8)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].legend(fontsize=9, facecolor=BG_COLOR, edgecolor=GRID_COLOR)
axes[0].grid(color=GRID_COLOR, alpha=0.4)
axes[0].set_axisbelow(True)

# ── Robustesse aux covariables ────────────────────────────────
axes[1].set_facecolor(BG_COLOR)
col_ate = [c for c in sensitivity_df.columns if 'ATE' in c or 'theta' in c.lower()][0]
y_pos = list(range(len(sensitivity_df)))
axes[1].barh(y_pos, sensitivity_df[col_ate],
             xerr=[sensitivity_df[col_ate] - sensitivity_df['IC95 low'],
                   sensitivity_df['IC95 high'] - sensitivity_df[col_ate]],
             color=COLORS['treated'], alpha=0.75, height=0.45,
             error_kw={'elinewidth': 1.8, 'capsize': 6, 'ecolor': TEXT_COLOR},
             edgecolor=BG_COLOR)
axes[1].scatter(sensitivity_df[col_ate], y_pos, color=TEXT_COLOR, zorder=5, s=50)
axes[1].axvline(0, color=ACCENT_COLOR, lw=1.8, ls='--', alpha=0.9)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(sensitivity_df['Ensemble de covariables'], fontsize=9, fontfamily=FONT_BODY)
axes[1].set_title('Robustesse à la spécification des covariables\n(AIPW, outcome = Dropout)',
                  fontsize=14, fontfamily=FONT_TITLE)
axes[1].set_xlabel('θ (ATE)', fontsize=12, labelpad=8)
axes[1].grid(axis='x', color=GRID_COLOR, alpha=0.4)
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()


**Interprétation du test placebo :** Sous l'hypothèse nulle (bourse assignée aléatoirement), la distribution des estimés est centrée sur zéro. Comm notre estimé réel $\theta$ tombe largement dans les queues de cette distribution, cela confirme que l'effet observé n'est pas dû au hasard.

**Interprétation de la robustesse aux covariables :** Si l'estimé reste stable quelle que soit la spécification des covariables choisie, cela suggère que le résultat ne dépend pas d'un choix arbitraire de contrôles. En revanche, lorsqu'on inclut les variables post-traitement (notes semestrielles), l'estimé change car on ferme les chemins causaux médiateurs - ce qui est attendu et confirme la validité du DAG.

## Synthèse Comparative des Estimateurs Causaux

Nous compilons ici l'ensemble des estimations causales obtenues par les différentes méthodes. Cette comparaison s'inscrit dans la logique de **triangulation des estimateurs** préconisée par la littérature empirique moderne : si des méthodes reposant sur des hypothèses différentes (sparsité pour le Post-Lasso, non-paramétrique pour le RF, double robustesse pour l'AIPW) convergent vers le même résultat, on peut être plus confiant dans la robustesse de la conclusion.

Les méthodes comparées se distinguent par leurs **hypothèses d'identification** :

| Méthode | Hypothèse sur les nuisances | Traitement hétérogène |
|---|---|---|
| Double Post-Lasso | Sparsité linéaire | Non (ATE constant) |
| DML-PLR (Lasso) | Sparsité linéaire | Non |
| DML-PLR (RF/GB) | Non-paramétrique | Non (ATE constant) |
| DML-IRM (RF) | Non-paramétrique | Oui (APE/ATE) |
| AIPW (RF) | Double robustesse | Oui (ATE) |

La convergence des estimés entre ces cinq méthodes constitue notre principal argument de robustesse.


In [ ]:
# ─── Tableau récapitulatif ──────────────────────────────────
summary_rows = [
    {'Méthode': 'Double Post-Lasso',    'Outcome': 'Dropout',
     'theta': coef_dpl, 'se': se_dpl, 'pval': pval_dpl,
     'ci_low': coef_dpl - 1.96 * se_dpl, 'ci_high': coef_dpl + 1.96 * se_dpl},
]
for row in results_dml:
    summary_rows.append({
        'Méthode': f"DML-PLR ({row['Learner']})",
        'Outcome': row['Outcome'],
        'theta': row['θ (ATE)'], 'se': row['SE'],
        'ci_low': row['IC95 low'], 'ci_high': row['IC95 high'],
        'pval': row['p-value']
    })
for outcome_name, res in irm_results.items():
    ci_vals = res['ci']
    summary_rows.append({
        'Méthode': 'DML-IRM (RF)',
        'Outcome': outcome_name,
        'theta': round(res['coef'], 4), 'se': round(res['se'], 4),
        'ci_low': round(ci_vals.iloc[0, 0], 4), 'ci_high': round(ci_vals.iloc[0, 1], 4),
        'pval': round(res['pval'], 4)
    })
for res in [aipw_dropout, aipw_graduate]:
    summary_rows.append({
        'Méthode': 'AIPW (RF)',
        'Outcome': res['outcome'],
        'theta': round(res['theta'], 4), 'se': round(res['se'], 4),
        'ci_low': round(res['ci_low'], 4), 'ci_high': round(res['ci_high'], 4),
        'pval': round(res['pval'], 4)
    })

summary_final = pd.DataFrame(summary_rows)
summary_final['Sig.'] = summary_final['pval'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.')))
summary_final['IC95'] = summary_final.apply(
    lambda r: f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]", axis=1)
summary_final['p-value'] = summary_final['pval'].apply(
    lambda p: f"{p:.4f}" if p >= 0.0001 else "<0.0001")

print("═"*80)
print("SYNTHÈSE DES ESTIMATEURS CAUSAUX")
print("═"*80)
print(summary_final[['Méthode', 'Outcome', 'theta', 'se', 'IC95', 'p-value', 'Sig.']].rename(
    columns={'theta': 'θ (ATE)', 'se': 'SE'}).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Forest Plot - Comparaison de tous les Estimateurs Causaux',
             fontsize=17, fontfamily=FONT_TITLE, y=1.01)

for ax, outcome, color in zip(axes,
                               ['Dropout', 'Graduate'],
                               [COLORS['dropout'], COLORS['graduate']]):
    ax.set_facecolor(BG_COLOR)
    sub = summary_final[summary_final['Outcome'] == outcome].reset_index(drop=True)
    if sub.empty:
        ax.axis('off')
        continue
    y_pos = list(range(len(sub)))
    ax.barh(y_pos, sub['theta'],
            xerr=[sub['theta'] - sub['ci_low'], sub['ci_high'] - sub['theta']],
            color=color, alpha=0.75, height=0.45,
            error_kw={'elinewidth': 1.8, 'capsize': 6, 'ecolor': TEXT_COLOR},
            edgecolor=BG_COLOR)
    ax.scatter(sub['theta'], y_pos, color=TEXT_COLOR, zorder=5, s=55)
    ax.axvline(0, color=ACCENT_COLOR, lw=2, ls='--', alpha=0.9)
    ax.set_yticks(y_pos)
    methode_col = [c for c in sub.columns if 'thode' in c or 'ethod' in c][0]
    ax.set_yticklabels(sub[methode_col], fontsize=10, fontfamily=FONT_BODY)
    ax.set_title(f'Outcome : {outcome}', fontsize=15, fontfamily=FONT_TITLE)
    ax.set_xlabel('ATE estimé (effet de la bourse)', fontsize=12, labelpad=8)
    ax.grid(axis='x', color=GRID_COLOR, alpha=0.4)
    ax.set_axisbelow(True)
    for i, row in sub.iterrows():
        ax.text(row['theta'], i + 0.30,
                f"θ={row['theta']:+.3f} {row['Sig.']}",
                ha='center', fontsize=9, color=TEXT_COLOR)

plt.tight_layout()
plt.show()


## Conclusion et Discussion

### Synthèse des résultats

L'ensemble de nos estimateurs causaux converge vers un résultat cohérent : **la bourse scolaire a un effet protecteur significatif contre le décrochage** et un effet positif sur les chances de diplomation.

- **Double Post-Lasso** : la sélection de variables via Lasso identifie les covariables les plus pertinentes pour prédire à la fois l'outcome et le traitement, garantissant que l'effet estimé n'est pas le fruit d'une omission de variables importantes.

- **DML-PLR** : les trois learners (Lasso, Random Forest, Gradient Boosting) convergent vers des estimés similaires, ce qui renforce la confiance dans la robustesse du résultat. L'utilisation du cross-fitting (5 folds) évite le biais de régularisation.

- **DML-IRM** : le modèle interactif, qui relaxe l'hypothèse de linéarité, confirme les résultats du PLR, suggérant que la relation entre bourse et outcomes ne dépend pas fortement d'interactions complexes avec les covariables.

- **AIPW** : l'estimateur doublement robuste confirme l'effet, avec en prime des scores individuels permettant l'analyse des effets hétérogènes.

### Effets hétérogènes et implications de politique publique

L'analyse HTE révèle que l'effet de la bourse n'est pas uniforme :
- Les étudiants **débiteurs** et ceux dont les **frais ne sont pas à jour** présentent un CATE particulièrement négatif sur le dropout - la bourse est plus efficace pour eux.
- Les étudiants **plus âgés** (> 25 ans) semblent moins bénéficier de la bourse, possiblement car leurs difficultés sont davantage liées à la conciliation vie professionnelle/études.

Le **policy learning** suggère qu'une redistribution des bourses vers les étudiants à fort CATE négatif permettrait d'améliorer l'efficacité de la politique, en ciblant ceux pour qui l'aide financière a le plus d'impact causal sur la rétention.

### Limites de l'étude

Notre analyse comporte plusieurs limites importantes :

1. **Hypothèse d'ignorabilité conditionnelle** : l'identification causale repose sur l'hypothèse que, conditionnellement aux covariables observées, l'assignation de la bourse est quasi-aléatoire. Or, des facteurs non observés comme la **motivation intrinsèque** ou le **capital social familial** pourraient simultanément influencer la probabilité d'obtenir une bourse et la réussite académique. Notre DAG modélise cette incertitude via le nœud U non observé.

2. **Données observationnelles** : en l'absence d'une expérience aléatoire (RCT), il est impossible d'établir avec certitude la causalité. Nos résultats constituent une estimation *conditionnellement* aux hypothèses d'identification.

3. **Données portugaises** : le dataset est issu d'une seule université portugaise. La généralisation à d'autres contextes nationaux ou institutionnels requiert prudence, notamment en ce qui concerne les mécanismes d'attribution des bourses et le système universitaire.

4. **SUTVA** : l'hypothèse de non-interférence entre étudiants est raisonnable à l'échelle individuelle, mais pourrait être violée si les bourses modifient l'environnement académique général (ex : effets de pairs).

5. **Variable de traitement binaire** : le montant de la bourse n'est pas connu, ce qui empêche d'analyser les effets dose-réponse et l'optimalité du montant alloué.

### Perspectives

Ce travail ouvre plusieurs pistes de recherche :
- Intégrer des données de panel permettant de suivre les mêmes étudiants dans le temps (*diff-in-diff* ou *event study*)
- Tester des variables instrumentales (ex : cutoffs d'éligibilité aux bourses) pour une identification plus robuste
- Étendre l'analyse à d'autres universités pour améliorer la généralisation externe